# P3.7 — Exploratory gate: synthetic scoring pass (FUS-GATE-001)

Runs the frozen seed-42 experts **P2.1 Semantic**, **P2.2 Sequence reference** and **P2.3 Sequence lightweight**
(P2-FREEZE-001) on the preregistered synthetic copies of the checkpoint-selection windows of each target fold
(CLEAN, FOREIGN_TARGET, SHUFFLED_CONTEXT). An expert scores only the copies whose input differs from CLEAN. Nothing is
trained and no label is read. P2.4 GTAT and the gate run on a local CPU. Each expert must first reproduce its stored
sanity score. The gate is EXPLORATORY: G2 stays PARTIAL. Spec: `configs/protocols/fus-gate-v1.yaml`.

## Setup

1. In Colab, choose **Runtime → Change runtime type** and select an **L4** (native BF16, at least 20 GiB).
2. Upload `seqlogad-code-gate-v1.zip` and `p3.7-synthetic-view-v1.zip` to `MyDrive/SeqLogAD/P3.7_Gate/uploads/`.
3. The seed-42 FINAL ZIPs and the data ZIPs must be where the P3.2 pass read them (`P2.1_Semantic/`, `P2.2_Sequence/`,
   `P2.3_Lightweight/`, `P2.4_GTAT/` under `MyDrive/SeqLogAD/`).
4. Keep the Hugging Face token in **Colab Secrets** as `HF_TOKEN` with notebook access. Never paste it into a cell.
5. Choose **Runtime → Run all**. Expect about 2 hours on an L4.

Progress is saved on Drive every 256 copies. If Colab disconnects, run the notebook again: finished copies and experts
are skipped. Download `MyDrive/SeqLogAD/P3.7_Gate/P3.7_synthetic_scores.zip` and its `.sha256` only after
`SAFE_TO_DISCONNECT_COLAB=YES`.


In [ ]:
# HUMAN CONFIG — fixed by FUS-GATE-001 (configs/protocols/fus-gate-v1.yaml); EXPLORATORY.
from pathlib import Path
import os, uuid
from google.colab import drive

drive.mount("/content/drive")

TASK_ID = "P3.7"
NOTEBOOK_STATUS = "READY"
BLOCKERS = []
TARGETS = ["BGL", "HDFS"]  # EXC-007: Hadoop/OpenStack are source-only
DRIVE_ROOT = Path("/content/drive/MyDrive/SeqLogAD/P3.7_Gate")
UPLOAD_ROOT = DRIVE_ROOT / "uploads"  # the P3.7 code ZIP and the synthetic view ZIP
# Data ZIPs come from the seed-42 upload folders; nothing is written there.
UPLOAD_ROOTS = [UPLOAD_ROOT] + [Path("/content/drive/MyDrive/SeqLogAD") / folder / "uploads" for folder in ("P2.2_Sequence", "P2.1_Semantic", "P2.4_GTAT")]
SEQLOGAD = Path("/content/drive/MyDrive/SeqLogAD")
FINAL_ZIPS = {  # seed-42 deliverables of P2-FREEZE-001
    "P2.1": SEQLOGAD / "P2.1_Semantic/P2.1/seqlogad_outputs_P2.1_S42_FINAL.zip",
    "P2.2": SEQLOGAD / "P2.2_Sequence/P2.2/seqlogad_outputs_P2.2_S42_FINAL.zip",
    "P2.3": SEQLOGAD / "P2.3_Lightweight/P2.3/seqlogad_outputs_P2.3_S42_FINAL.zip",
}
PASS_TASKS = "P2.3,P2.2,P2.1"  # P2.4 and the gate run on a local CPU
PERSISTENT_ROOT = DRIVE_ROOT
# Keep Python, extracted source, tokenizer cache and temporary outputs on Colab's local disk.
STATE_ROOT = Path("/content/SeqLogAD")
HF_HOME = STATE_ROOT / "hf-cache"
VENV = STATE_ROOT / "venv-py312"
RUNTIME_ROOT = STATE_ROOT / "runtime" / "gate-62ca1fe59cad"
REPO_ROOT = str(RUNTIME_ROOT / "seqlogad_code_gate")
VIEW_ROOT = str(RUNTIME_ROOT)  # holds synthetic/FUS-GATE-001/<fold>/{mutations,rows}.parquet
DATA_ROOT = str(RUNTIME_ROOT / "phase2_sequence")
SEMANTIC_ROOT = str(RUNTIME_ROOT / "phase2")
GRAPH_ROOT = str(RUNTIME_ROOT / "phase2_graph")
PASS_ROOT = DRIVE_ROOT / "synthetic_scores"  # chunks and receipts persist on Drive, so a rerun resumes
WORK_ROOT = STATE_ROOT / "p3.7-work"
SESSION_ROOT = Path("/content/seqlogad_sessions") / uuid.uuid4().hex
EXPECTED_CODE_ZIP_SHA256 = "62ca1fe59cadb36f086c2ffc52ee01a4e3c159419569f0fae5b7bbac63c9d054"
EXPECTED_VIEW_ZIP_SHA256 = "d67204e8cd0c4e1015ef907778f29d7c52fe944f702d5676bedc99bf1f4399db"
EXPECTED_DATA_ZIP_SHA256 = "4ed76b92e4b001138b28b4b3b4d3017d6ff7f30dd31e8cca0649bfdf34ede781"
EXPECTED_SEMANTIC_ZIP_SHA256 = "ebcfdfd6eb7b1607eb6d57e60fae88456d03bcb8ae2c6688ea398042c856224b"
EXPECTED_BUNDLE_SHA256 = "1268b669a8b1b474658cfa7daa6556c987b0f31f07e0101ed30af250a8d7f129"
EXPECTED_GRAPH_ZIP_SHA256 = "595b5dc7ec1b96abc80b4722b5f39dcf1c5653ec25aefebcce7ee74dbf36f9ac"
EXPECTED_GRAPH_MANIFEST_SHA256 = "8918172b4376fd822ec812e9b46b2725f83b517a43c94f3efe16f55623c9d059"
MIN_VRAM_GIB = 20

assert TARGETS == ["BGL", "HDFS"]
assert all(path.is_file() for path in FINAL_ZIPS.values()), "A seed-42 FINAL ZIP is missing on Drive"
assert NOTEBOOK_STATUS == "READY", "Notebook is fail-closed: " + ", ".join(BLOCKERS)
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
for directory in (STATE_ROOT, PERSISTENT_ROOT, HF_HOME, SESSION_ROOT, PASS_ROOT, WORK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
print({"session_root": str(SESSION_ROOT), "persistent_root": str(PERSISTENT_ROOT),
       "upload_folders": [str(root) for root in UPLOAD_ROOTS]})


In [ ]:
# Colab normally includes uv; install it only if absent.
import sys, shutil, subprocess

uv_real = shutil.which("uv")
if not uv_real:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.8.17"], check=True)
    uv_real = shutil.which("uv")
assert uv_real, "uv installation failed"
uv_expected = Path(sys.executable).parent / "uv"
if not uv_expected.exists():
    uv_expected.symlink_to(uv_real)
print("uv:", uv_real)


## Verify uploads, extract atomically and build the frozen runtime


In [ ]:
from pathlib import PurePosixPath
import hashlib, json, shutil, stat, subprocess, sys, zipfile

def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

def locate(expected):
    # First match wins, so a ZIP copied into both upload folders is not ambiguous.
    for root in UPLOAD_ROOTS:
        for path in (sorted(root.glob("*.zip")) if root.is_dir() else []):
            if digest(path) == expected:
                return path
    raise AssertionError(f"Put the ZIP with SHA-256 {expected} in one of {[str(r) for r in UPLOAD_ROOTS]}")

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as handle:
        infos = handle.infolist()
        names = [item.filename for item in infos]
        assert len(names) == len(set(names)), "Duplicate ZIP member"
        for item in infos:
            member = PurePosixPath(item.filename)
            assert not member.is_absolute() and ".." not in member.parts and "\\" not in item.filename
            assert ((item.external_attr >> 16) & 0o170000) != stat.S_IFLNK, "ZIP symlink forbidden"
        assert handle.testzip() is None, "ZIP CRC failure"
        handle.extractall(destination)

code_zip = locate(EXPECTED_CODE_ZIP_SHA256)
data_zip = locate(EXPECTED_DATA_ZIP_SHA256)
semantic_zip = locate(EXPECTED_SEMANTIC_ZIP_SHA256)
graph_zip = locate(EXPECTED_GRAPH_ZIP_SHA256)
view_zip = locate(EXPECTED_VIEW_ZIP_SHA256)
marker = RUNTIME_ROOT / "EXTRACTION.json"
expected_marker = {
    "code_sha256": EXPECTED_CODE_ZIP_SHA256,
    "data_sha256": EXPECTED_DATA_ZIP_SHA256,
    "semantic_sha256": EXPECTED_SEMANTIC_ZIP_SHA256,
    "graph_sha256": EXPECTED_GRAPH_ZIP_SHA256,
    "view_sha256": EXPECTED_VIEW_ZIP_SHA256,
}
if RUNTIME_ROOT.exists():
    assert marker.is_file(), f"Unverified runtime directory exists: {RUNTIME_ROOT}"
    assert json.loads(marker.read_text()) == expected_marker, "Runtime extraction identity mismatch"
else:
    partial = RUNTIME_ROOT.with_name(RUNTIME_ROOT.name + ".partial")
    if partial.exists():
        shutil.rmtree(partial)
    partial.mkdir(parents=True)
    safe_extract(code_zip, partial)
    safe_extract(data_zip, partial)
    safe_extract(semantic_zip, partial)
    safe_extract(graph_zip, partial)
    safe_extract(view_zip, partial)
    assert (partial / "seqlogad_code_gate").is_dir()
    assert (partial / "phase2_sequence/manifests/bundle.json").is_file()
    assert (partial / "phase2/manifests/bundle.json").is_file()
    assert (partial / "phase2_graph/manifests/graph-bundle.json").is_file()
    assert (partial / "synthetic/FUS-GATE-001").is_dir()
    (partial / "EXTRACTION.json").write_text(json.dumps(expected_marker, sort_keys=True) + "\n")
    partial.rename(RUNTIME_ROOT)

assert Path(REPO_ROOT).is_dir()
assert Path(DATA_ROOT, "manifests/bundle.json").is_file()
assert digest(Path(DATA_ROOT) / "manifests/bundle.json") == EXPECTED_BUNDLE_SHA256
assert digest(Path(GRAPH_ROOT) / "manifests/graph-bundle.json") == EXPECTED_GRAPH_MANIFEST_SHA256
print({"code_zip": str(code_zip), "data_zip": str(data_zip), "semantic_zip": str(semantic_zip),
       "extraction": "PASS"})

def isolated_env():
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    return env

def run_live(args, check=True):
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=isolated_env()) as process:
        for line in process.stdout:
            token = os.environ.get("HF_TOKEN")
            if token:
                line = line.replace(token, "[TOKEN_HIDDEN]")
            print(line, end="", flush=True)
        code = process.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, args)
    return code

run_live(["nvidia-smi"])
PYTHON = str(VENV / "bin/python")
if not Path(PYTHON).is_file():
    if sys.version_info[:2] == (3, 12):
        run_live([sys.executable, "-m", "pip", "install", "virtualenv==20.31.2"])
        run_live([sys.executable, "-m", "virtualenv", "--python", sys.executable, str(VENV)])
    else:
        run_live([sys.executable, "-m", "pip", "install", "uv==0.8.17"])
        uv = str(Path(sys.executable).parent / "uv")
        assert Path(uv).is_file(), "uv bootstrap executable missing"
        run_live([uv, "python", "install", "3.12"])
        run_live([uv, "venv", "--python", "3.12", "--seed", str(VENV)])

run_live([PYTHON, "-c", "import sys; assert sys.version_info[:2] == (3, 12); print(sys.version)"])
run_live([PYTHON, "-m", "pip", "install", "torch==2.9.1", "--index-url",
          "https://download.pytorch.org/whl/cu128"])
run_live([PYTHON, "-m", "pip", "install", "-e", REPO_ROOT, "packaging>=24,<27"])
run_live([PYTHON, "-c", "from seqlogad.semantic.runtime import install_missing; install_missing(" + repr(REPO_ROOT) + ")"])
hardware_check = f"""
import json, torch
assert torch.cuda.is_available(), "CUDA is unavailable"
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info(0)
native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
capability = torch.cuda.get_device_capability(0)
assert total >= {MIN_VRAM_GIB} * 1024**3, f"Expected at least {MIN_VRAM_GIB} GiB VRAM; got {{total / 1024**3:.1f}} GiB on {{name}}"
assert native_bf16, f"The P3.7 pass of P2.1/P2.2 requires native BF16 (L4/A100/H100/G4); {{name}} does not support it"
print(json.dumps({{"gpu": name, "compute_capability": capability, "free_vram_bytes": free, "total_vram_bytes": total,
                  "native_bf16": native_bf16, "torch": torch.__version__,
                  "cuda": torch.version.cuda}}, indent=2))
"""
run_live([PYTHON, "-c", hardware_check])
run_live([PYTHON, "-m", "seqlogad.sequence.cli", "environment", "--repo-root", REPO_ROOT])


## Secure Hugging Face authentication


In [ ]:
# Read the token from Colab Secrets; it is never printed or written to Drive.
from google.colab import userdata

token = os.environ.get("HF_TOKEN")
if not token:
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
assert token, (
    "Add HF_TOKEN in Colab Secrets, enable notebook access, then rerun this cell"
)
run_live([PYTHON, "-m", "seqlogad.sequence.cli", "auth", "--repo-root", REPO_ROOT])


## Synthetic scoring pass (resumable; progress persists on Drive)

In [ ]:
# Scores the synthetic copies; one progress line per chunk of 256 copies.
args = [
    PYTHON, "-m", "seqlogad.adaptation.synthetic", "score",
    "--repo-root", REPO_ROOT,
    "--view-root", VIEW_ROOT,
    "--data-root", DATA_ROOT,
    "--semantic-root", SEMANTIC_ROOT,
    "--graph-root", GRAPH_ROOT,
    "--output-root", str(PASS_ROOT),
    "--work-root", str(WORK_ROOT),
    "--tasks", PASS_TASKS,
]
for task, path in FINAL_ZIPS.items():
    args += ["--final", f"{task}={path}"]
run_live(args, check=True)


## Final persistent-integrity and safe-stop gate


In [ ]:
# Every expert and fold must have a complete pass and a passing sanity check; package them for download.
experts = {"P2.1": "SEMANTIC_LLAMA", "P2.2": "SEQUENCE_LLAMA_REFERENCE", "P2.3": "SEQUENCE_LIGHTWEIGHT"}
package = DRIVE_ROOT / "P3.7_synthetic_scores.zip"
with zipfile.ZipFile(SESSION_ROOT / package.name, "w", zipfile.ZIP_DEFLATED) as handle:
    for target in TARGETS:
        fold = f"FOLD-TARGET-ARCH-{target}"
        for task, expert in experts.items():
            receipt = json.loads((PASS_ROOT / fold / f"{expert}.receipt.json").read_text())
            assert receipt["sanity_reproduction"]["pass"] and not receipt["labels_read"]
            path = PASS_ROOT / fold / f"{expert}.jsonl"
            assert receipt["scores_sha256"] == digest(path), f"{task} {target} {path.name} changed"
            handle.write(path, f"synthetic_scores/{fold}/{path.name}")
            handle.write(PASS_ROOT / fold / f"{expert}.receipt.json", f"synthetic_scores/{fold}/{expert}.receipt.json")
            print(task, target, receipt["copies"], receipt["status_counts"])
shutil.copyfile(SESSION_ROOT / package.name, package)
package_hash = digest(package)
Path(str(package) + ".sha256").write_text(f"{package_hash}  {package.name}\n")
shutil.rmtree(SESSION_ROOT)
print("====================================")
print("SEQLOGAD COLAB EXECUTION SUMMARY - P3.7 SYNTHETIC SCORING")
print(f"SYNTHETIC_SCORES_ZIP={package}")
print(f"SYNTHETIC_SCORES_ZIP_SHA256={package_hash}")
print("PERSISTENT_STORAGE_VERIFIED=YES")
print("SAFE_TO_DISCONNECT_COLAB=YES")
print("====================================")
